# 🧹 2. Data Cleaning and Shared Experiment Preparation

This notebook prepares the Anime Recommendations Database for the content-based recommender.

It answers one question:

> What must happen to the raw anime and rating tables before content features and user profiles are built?

This notebook stops at the preprocessing boundary. It does **not** create TF-IDF vectors, user profiles, similarity scores, or recommendations.

## 🗺️ Pipeline

```text
Raw rating.csv
      ↓
Remove ambiguous duplicate user-anime pairs
      ↓
Define ratings 7–10 as positive
      ↓
Construct the iterative positive 5-core
      ↓
Create training, validation, and test positives
      ↓
Create fixed evaluation candidates
      ↓
Join all warm anime to metadata
      ↓
Clean handoff to content-based filtering
```

Low ratings and unrated watches are not positive preferences. They remain observed interactions so they cannot be sampled as unseen candidates.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
POSITIVE_THRESHOLD = 7
CORE_SIZE = 5
NEGATIVE_COUNT = 99

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "data.py").exists():
            return path
    raise FileNotFoundError("Run this notebook inside the dam repository.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    assert_split_integrity,
    build_observed_sets,
    iterative_positive_k_core,
    leave_two_out_split,
    load_anime_ratings,
    positive_interactions,
    remove_ambiguous_pairs,
    sample_evaluation_candidates,
)

print("Repository:", REPO_ROOT)

## 1️⃣ Load the two raw tables

Set the environment variable `ANIME_DATASET_DIR` to a folder containing:

- `anime.csv`
- `rating.csv`

If it is not set, the notebook downloads Kaggle dataset version 1.

In [ ]:
def resolve_dataset_dir() -> Path:
    configured = os.getenv("ANIME_DATASET_DIR")
    if configured:
        path = Path(configured).expanduser().resolve()
    else:
        try:
            import kagglehub
        except ImportError as exc:
            raise ImportError(
                "Install kagglehub or set ANIME_DATASET_DIR to the downloaded dataset folder."
            ) from exc
        path = Path(
            kagglehub.dataset_download("CooperUnion/anime-recommendations-database")
        )

    missing = [
        filename
        for filename in ("anime.csv", "rating.csv")
        if not (path / filename).exists()
    ]
    if missing:
        raise FileNotFoundError(f"Missing files in {path}: {missing}")
    return path

DATASET_DIR = resolve_dataset_dir()
anime_raw = pd.read_csv(DATASET_DIR / "anime.csv")
ratings_raw = load_anime_ratings(DATASET_DIR / "rating.csv")

print("Dataset folder:", DATASET_DIR)
print(f"Raw anime rows: {len(anime_raw):,}")
print(f"Raw rating rows: {len(ratings_raw):,}")
display(anime_raw.head())
display(ratings_raw.head())

## 2️⃣ Understand the rating meanings

For this experiment:

| Rating | Meaning |
| ---: | --- |
| 7–10 | Positive preference |
| 1–6 | Observed, but not positive |
| -1 | Watched, but not rated |

Ratings below 7 and `-1` will not build the content profile. They are retained as observed history so the corresponding anime cannot later be called “unseen.”

In [ ]:
rating_summary = (
    ratings_raw["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="rows")
)
rating_summary["experiment_role"] = np.select(
    [
        rating_summary["rating"].eq(-1),
        rating_summary["rating"].between(1, 6),
        rating_summary["rating"].between(7, 10),
    ],
    [
        "observed, unrated",
        "observed, not positive",
        "positive",
    ],
    default="unexpected",
)
display(rating_summary)
assert ratings_raw["rating"].between(-1, 10).all()
assert not ratings_raw[["user_id", "anime_id", "rating"]].isna().any().any()

## 3️⃣ Remove ambiguous duplicate pairs

A user-anime pair should describe one preference.

If a pair appears more than once, all its rows are removed. Keeping the first, last, or average would invent a rule for deciding which duplicated preference is correct.

In [ ]:
duplicate_rows = ratings_raw.duplicated(
    ["user_id", "anime_id"], keep=False
)
duplicate_examples = ratings_raw.loc[duplicate_rows].sort_values(
    ["user_id", "anime_id", "rating"]
)

deduplicated_ratings, ambiguous_pair_count = remove_ambiguous_pairs(ratings_raw)

print(f"Ambiguous user-anime pairs: {ambiguous_pair_count:,}")
print(f"Rows removed: {len(ratings_raw) - len(deduplicated_ratings):,}")
display(duplicate_examples)

assert ambiguous_pair_count == 7
assert len(ratings_raw) - len(deduplicated_ratings) == 14
assert not deduplicated_ratings.duplicated(["user_id", "anime_id"]).any()

## 4️⃣ Extract positive interactions

Only ratings of 7–10 enter the positive preference dataset.

The other interactions remain in `deduplicated_ratings`. They will later prevent watched or rated anime from being sampled as unseen negatives.

In [ ]:
positives_before_core = positive_interactions(
    deduplicated_ratings,
    positive_threshold=POSITIVE_THRESHOLD,
)

print(f"Deduplicated observed interactions: {len(deduplicated_ratings):,}")
print(f"Positive interactions before the 5-core: {len(positives_before_core):,}")
print(
    "Positive users:",
    f"{positives_before_core['user_id'].nunique():,}",
)
print(
    "Positive anime:",
    f"{positives_before_core['anime_id'].nunique():,}",
)

assert positives_before_core["rating"].between(7, 10).all()
assert not positives_before_core.duplicated(["user_id", "anime_id"]).any()

## 5️⃣ Construct the iterative positive 5-core

Every retained user and anime must have at least five positive interactions.

This must be iterative:

```text
Remove sparse users or anime
           ↓
Their removal changes other interaction counts
           ↓
Repeat until every remaining user and anime has degree ≥ 5
```

This produces a stable warm-start population suitable for leave-two-out evaluation.

In [ ]:
positive_core, core_stats = iterative_positive_k_core(
    positives_before_core,
    min_degree=CORE_SIZE,
)

core_summary = pd.DataFrame({
    "stage": ["before 5-core", "after 5-core"],
    "positive rows": [
        core_stats.input_positive_count,
        core_stats.output_positive_count,
    ],
    "users": [
        core_stats.input_user_count,
        core_stats.output_user_count,
    ],
    "anime": [
        core_stats.input_item_count,
        core_stats.output_item_count,
    ],
})
display(core_summary)
print("Filtering iterations:", core_stats.iterations)

user_degrees = positive_core.groupby("user_id")["anime_id"].size()
item_degrees = positive_core.groupby("anime_id")["user_id"].size()
assert int(user_degrees.min()) >= CORE_SIZE
assert int(item_degrees.min()) >= CORE_SIZE
assert positive_core["user_id"].nunique() == 60_384

## 6️⃣ Create training, validation, and test positives

For each user:

- one positive is hidden for validation
- one positive is hidden for test
- all other positives become training data

The split uses seed 42. The preparation code repairs the split when necessary so every held-out anime still appears in training.

In [ ]:
split = leave_two_out_split(positive_core, seed=SEED)

train_positives = split.train
validation_positives = split.validation
test_positives = split.test

eligible_users = sorted(int(value) for value in train_positives["user_id"].unique())
warm_items = sorted(int(value) for value in train_positives["anime_id"].unique())

split_summary = pd.DataFrame({
    "partition": ["training", "validation", "test"],
    "positive rows": [
        len(train_positives),
        len(validation_positives),
        len(test_positives),
    ],
    "users": [
        train_positives["user_id"].nunique(),
        validation_positives["user_id"].nunique(),
        test_positives["user_id"].nunique(),
    ],
    "anime": [
        train_positives["anime_id"].nunique(),
        validation_positives["anime_id"].nunique(),
        test_positives["anime_id"].nunique(),
    ],
})
display(split_summary)

assert len(eligible_users) == 60_384
assert len(warm_items) == 7_223
assert len(train_positives) == 5_087_394
assert len(validation_positives) == 60_384
assert len(test_positives) == 60_384
assert validation_positives["user_id"].value_counts().eq(1).all()
assert test_positives["user_id"].value_counts().eq(1).all()
assert set(validation_positives["anime_id"]).issubset(warm_items)
assert set(test_positives["anime_id"]).issubset(warm_items)

### Concrete user example

The next cell shows one user's actual split.

Only the training rows are visible when the content-based system builds that user's profile. The validation and test positives are hidden answers.

In [ ]:
example_user = eligible_users[0]

example_train = (
    train_positives.loc[train_positives["user_id"].eq(example_user)]
    .assign(partition="training")
)
example_validation = (
    validation_positives.loc[validation_positives["user_id"].eq(example_user)]
    .assign(partition="validation")
)
example_test = (
    test_positives.loc[test_positives["user_id"].eq(example_user)]
    .assign(partition="test")
)

example_split = pd.concat(
    [example_train, example_validation, example_test],
    ignore_index=True,
).merge(
    anime_raw[["anime_id", "name"]],
    on="anime_id",
    how="left",
)
display(
    example_split[
        ["partition", "user_id", "anime_id", "name", "rating"]
    ].sort_values(["partition", "rating"], ascending=[True, False])
)

## 7️⃣ Record every observed anime for each eligible user

The profile uses positive training rows only.

Candidate sampling needs a broader history: every anime the user rated or marked as watched. This prevents low-rated and unrated anime from being mislabeled as unseen negatives.

In [ ]:
observed_by_user = build_observed_sets(
    deduplicated_ratings,
    eligible_users,
)

example_observed = observed_by_user[example_user]
example_positive_train = set(
    train_positives.loc[
        train_positives["user_id"].eq(example_user),
        "anime_id",
    ].astype(int)
)

print(f"User {example_user} observed anime: {len(example_observed):,}")
print(f"User {example_user} positive training anime: {len(example_positive_train):,}")
print(
    "Observed but not in the positive training profile:",
    f"{len(example_observed - example_positive_train):,}",
)
assert example_positive_train.issubset(example_observed)

## 8️⃣ Create fixed validation and test candidates

Each user receives exactly 100 candidates:

```text
1 hidden positive + 99 unseen warm anime
```

A negative candidate must not occur anywhere in that user's observed history.

Validation sampling uses seed 42. Test sampling uses seed 43.

In [ ]:
validation_candidates = sample_evaluation_candidates(
    held_out=validation_positives,
    observed_by_user=observed_by_user,
    warm_items=warm_items,
    negative_count=NEGATIVE_COUNT,
    seed=SEED,
)
test_candidates = sample_evaluation_candidates(
    held_out=test_positives,
    observed_by_user=observed_by_user,
    warm_items=warm_items,
    negative_count=NEGATIVE_COUNT,
    seed=SEED + 1,
)

candidate_summary = pd.DataFrame({
    "split": ["validation", "test"],
    "users": [
        validation_candidates["user_id"].nunique(),
        test_candidates["user_id"].nunique(),
    ],
    "candidate rows": [
        len(validation_candidates),
        len(test_candidates),
    ],
    "positive rows": [
        int(validation_candidates["label"].sum()),
        int(test_candidates["label"].sum()),
    ],
    "negative rows": [
        int(validation_candidates["label"].eq(0).sum()),
        int(test_candidates["label"].eq(0).sum()),
    ],
})
display(candidate_summary)

assert_split_integrity(
    split,
    validation_candidates,
    test_candidates,
    observed_by_user,
)
assert len(validation_candidates) == 60_384 * 100
assert len(test_candidates) == 60_384 * 100
print("✅ Candidate sets and split integrity are valid.")

In [ ]:
example_candidates = validation_candidates.loc[
    validation_candidates["user_id"].eq(example_user)
].merge(
    anime_raw[["anime_id", "name"]].rename(columns={"anime_id": "item_id"}),
    on="item_id",
    how="left",
)

display(example_candidates.head(10))
print(
    "Positive candidates:",
    int(example_candidates["label"].sum()),
)
print(
    "Negative candidates:",
    int(example_candidates["label"].eq(0).sum()),
)

## 9️⃣ Join all warm anime to metadata

The content model needs one metadata row for every warm anime.

We use the raw anime table rather than dropping missing genres. Anime 33242 has no genre but remains a valid shared candidate with a title, type, and episode count.

In [ ]:
anime_metadata = (
    anime_raw
    .drop_duplicates("anime_id", keep="first")
    .set_index("anime_id")
    .reindex(warm_items)
    .reset_index()
)

metadata_coverage = pd.Series({
    "warm anime": len(warm_items),
    "metadata rows": len(anime_metadata),
    "missing names": int(anime_metadata["name"].isna().sum()),
    "missing genres": int(anime_metadata["genre"].isna().sum()),
    "missing types": int(anime_metadata["type"].isna().sum()),
    "missing episodes": int(
        pd.to_numeric(anime_metadata["episodes"], errors="coerce").isna().sum()
    ),
})
display(metadata_coverage.to_frame("count"))

missing_genre_rows = anime_metadata.loc[
    anime_metadata["genre"].isna(),
    ["anime_id", "name", "genre", "type", "episodes"],
]
display(missing_genre_rows)

assert len(anime_metadata) == 7_223
assert anime_metadata["anime_id"].notna().all()
assert anime_metadata["name"].notna().all()
assert set(missing_genre_rows["anime_id"]) == {33242}

## 🔍 10. Final integrity report

These checks define the exact boundary before content-based filtering begins.

In [ ]:
handoff_summary = pd.Series({
    "eligible users": len(eligible_users),
    "warm anime": len(warm_items),
    "training positives": len(train_positives),
    "validation positives": len(validation_positives),
    "test positives": len(test_positives),
    "validation candidates": len(validation_candidates),
    "test candidates": len(test_candidates),
    "metadata rows": len(anime_metadata),
})
handoff_display = handoff_summary.to_frame("count")
handoff_display["count"] = handoff_display["count"].map(
    lambda value: f"{int(value):,}"
)
display(handoff_display)

assert not train_positives.duplicated(["user_id", "anime_id"]).any()
assert set(validation_positives["anime_id"]).issubset(
    set(train_positives["anime_id"])
)
assert set(test_positives["anime_id"]).issubset(
    set(train_positives["anime_id"])
)
assert set(validation_candidates["item_id"]).issubset(warm_items)
assert set(test_candidates["item_id"]).issubset(warm_items)
assert set(anime_metadata["anime_id"]) == set(warm_items)

print("✅ Data is ready for content-based feature construction.")

## 📤 Handoff to the content-based notebook

The following in-memory objects are the preprocessing outputs:

| Object | Purpose |
| --- | --- |
| `train_positives` | Build rating-weighted user profiles |
| `validation_positives` | Hidden validation answers |
| `test_positives` | Hidden test answers |
| `validation_candidates` | Select C1–C4 |
| `test_candidates` | Evaluate the selected configuration once |
| `observed_by_user` | Confirm negative candidates are truly unobserved |
| `eligible_users` | Shared user universe |
| `warm_items` | Shared anime universe |
| `anime_metadata` | Build genre, type, episode, and title vectors |

The next stage may transform `anime_metadata` into content vectors. It must not change the shared users, warm anime, splits, or candidate rows.